In [1]:
import requests
import time
import pandas as pd

# --- Konfigurasi ---
BASE = "https://api.twitterapi.io/twitter/tweet/advanced_search"
HEAD = {"X-API-Key": "xxxxxxx"}  # Ganti dengan key aktif
query = "(pasar modal OR saham OR bursa OR investasi) since_time:1754006400 until_time:1756684800 -filter:retweets"
TARGET_JUMLAH = 1000
MAX_RETRIES = 5


def main():
    semua = []
    seen = set()
    cursor = ""
    retry_count = 0
    last_save_count = 0

    session = requests.Session()
    session.headers.update(HEAD)

    print("[INFO] Memulai ekstraksi data IHSG...")

    while len(semua) < TARGET_JUMLAH:
        params = {"query": query, "queryType": "Latest"}
        if cursor:
            params["cursor"] = cursor

        try:
            resp = session.get(BASE, params=params, timeout=15)
            if resp.status_code != 200:
                if resp.status_code == 429:
                    retry_after = resp.headers.get("Retry-After")
                    wait = (
                        int(retry_after)
                        if retry_after and str(retry_after).isdigit()
                        else 15
                    )
                    print(f"[RATE-LIMIT] Tunggu {wait}s...")
                    time.sleep(wait)
                    continue
                retry_count += 1
                print(
                    f"[WARNING] Error {resp.status_code} ({retry_count}/{MAX_RETRIES})"
                )
                time.sleep(10)
                if retry_count >= MAX_RETRIES:
                    print("[ERROR] Max retries. Stop.")
                    break
                continue
            r = resp.json()
            retry_count = 0
        except Exception as e:
            retry_count += 1
            print(f"[ERROR] {e} ({retry_count}/{MAX_RETRIES})")
            time.sleep(5)
            if retry_count >= MAX_RETRIES:
                print("[ERROR] Connection failed. Stop.")
                break
            continue

        tweets = r.get("tweets", [])
        if not tweets:
            print("[INFO] No more tweets.")
            break

        for t in tweets:
            tid = t.get("id")
            if not tid or tid in seen:
                continue
            seen.add(tid)
            user = t.get("author") or t.get("user") or {}

            # ---- EKSTRAKSI KONSISTEN ----
            username = (
                t.get("username") or user.get("userName") or user.get("screen_name")
            )
            name = t.get("name") or user.get("name")
            followers = (
                t.get("followers")
                if t.get("followers") is not None
                else user.get("followers")
            )
            verified = (
                t.get("verified")
                if t.get("verified") is not None
                else user.get("isBlueVerified")
            )

            row = {
                "id": tid,
                "created_at": t.get("createdAt"),
                "username": username,
                "name": name,
                "followers": followers,
                "verified": verified,
                "text": t.get("text"),
                "views": t.get("views"),
                "likes": t.get("likeCount"),
                "replies": t.get("replyCount"),
            }
            semua.append(row)

            if len(semua) >= TARGET_JUMLAH:
                break

        # Auto-save dengan encoding UTF-8-SIG untuk kompatibilitas penuh
        if len(semua) - last_save_count >= 100:
            pd.DataFrame(semua).to_csv(
                "dataset_ihsg_backup .csv", index=False, encoding="utf-8-sig"
            )
            print(f"[AUTO-SAVE] {len(semua)} data backed up.")
            last_save_count = len(semua)

        print(f"[PROGRESS] {len(semua)} / {TARGET_JUMLAH}")

        if not r.get("has_next_page"):
            break
        cursor = r.get("next_cursor")
        if not cursor:
            break
        time.sleep(2)

    # Finalisasi
    df = pd.DataFrame(semua)
    if not df.empty:
        df = df.drop_duplicates(subset=["id"])
        df.to_csv("dataset_ihsg_murni 2.csv", index=False, encoding="utf-8-sig")
        print(f"\n[SUCCESS] {len(df)} unique tweets saved.")
    else:
        print("\n[WARNING] No data.")


if __name__ == "__main__":
    main()


[INFO] Memulai ekstraksi data IHSG...
[WARNING] Error 401 (1/5)
[WARNING] Error 401 (2/5)
[WARNING] Error 401 (3/5)
[WARNING] Error 401 (4/5)
[WARNING] Error 401 (5/5)
[ERROR] Max retries. Stop.

[WARNING] No data.


Klasifikasi untuk setiap periode dan kalimat yang digunakan:
"pasar modal" OR IHSG OR bursa OR investasi OR BEI

In [ ]:
import requests
import time
import pandas as pd

# --- Konfigurasi ---
BASE = "https://api.twitterapi.io/twitter/tweet/advanced_search"
HEAD = {"X-API-Key": "xxxx"}  # Ganti dengan key aktif
query = "(saham OR bursa OR investasi OR BEI) lang:in since_time:1748736000 until_time:1759190399  -filter:retweets"
TARGET_JUMLAH = 5000
MAX_RETRIES = 5


def main():
    semua = []
    seen = set()
    cursor = ""
    retry_count = 0
    last_save_count = 0

    session = requests.Session()
    session.headers.update(HEAD)

    print("[INFO] Memulai ekstraksi data IHSG...")

    while len(semua) < TARGET_JUMLAH:
        params = {"query": query, "queryType": "Latest"}
        if cursor:
            params["cursor"] = cursor

        try:
            resp = session.get(BASE, params=params, timeout=15)
            if resp.status_code != 200:
                if resp.status_code == 429:
                    retry_after = resp.headers.get("Retry-After")
                    wait = (
                        int(retry_after)
                        if retry_after and str(retry_after).isdigit()
                        else 15
                    )
                    print(f"[RATE-LIMIT] Tunggu {wait}s...")
                    time.sleep(wait)
                    continue
                retry_count += 1
                print(
                    f"[WARNING] Error {resp.status_code} ({retry_count}/{MAX_RETRIES})"
                )
                time.sleep(10)
                if retry_count >= MAX_RETRIES:
                    print("[ERROR] Max retries. Stop.")
                    break
                continue
            r = resp.json()
            retry_count = 0
        except Exception as e:
            retry_count += 1
            print(f"[ERROR] {e} ({retry_count}/{MAX_RETRIES})")
            time.sleep(5)
            if retry_count >= MAX_RETRIES:
                print("[ERROR] Connection failed. Stop.")
                break
            continue

        tweets = r.get("tweets", [])
        if not tweets:
            print("[INFO] No more tweets.")
            break

        for t in tweets:
            tid = t.get("id")
            if not tid or tid in seen:
                continue
            seen.add(tid)
            user = t.get("author") or t.get("user") or {}

            # ---- EKSTRAKSI KONSISTEN ----
            username = (
                t.get("username") or user.get("userName") or user.get("screen_name")
            )
            name = t.get("name") or user.get("name")
            followers = (
                t.get("followers")
                if t.get("followers") is not None
                else user.get("followers")
            )
            verified = (
                t.get("verified")
                if t.get("verified") is not None
                else user.get("isBlueVerified")
            )

            row = {
                "id": tid,
                "created_at": t.get("createdAt"),
                "username": username,
                "name": name,
                "followers": followers,
                "verified": verified,
                "text": t.get("text"),
                "views": t.get("views"),
                "likes": t.get("likeCount"),
                "replies": t.get("replyCount"),
            }
            semua.append(row)

            if len(semua) >= TARGET_JUMLAH:
                break

        # Auto-save dengan encoding UTF-8-SIG untuk kompatibilitas penuh
        if len(semua) - last_save_count >= 100:
            pd.DataFrame(semua).to_csv(
                "dataset_ihsg_backup 2 .csv", index=False, encoding="utf-8-sig"
            )
            print(f"[AUTO-SAVE] {len(semua)} data backed up.")
            last_save_count = len(semua)

        print(f"[PROGRESS] {len(semua)} / {TARGET_JUMLAH}")

        if not r.get("has_next_page"):
            break
        cursor = r.get("next_cursor")
        if not cursor:
            break
        time.sleep(2)

    # Finalisasi
    df = pd.DataFrame(semua)
    if not df.empty:
        df = df.drop_duplicates(subset=["id"])
        df.to_csv(
            "dataset_ihsg_setelah agustus 2 .csv", index=False, encoding="utf-8-sig"
        )
        print(f"\n[SUCCESS] {len(df)} unique tweets saved.")
    else:
        print("\n[WARNING] No data.")


if __name__ == "__main__":
    main()

[INFO] Memulai ekstraksi data IHSG...
[PROGRESS] 20 / 5000
[PROGRESS] 40 / 5000
[PROGRESS] 60 / 5000
[PROGRESS] 80 / 5000
[AUTO-SAVE] 100 data backed up.
[PROGRESS] 100 / 5000
[PROGRESS] 120 / 5000
[PROGRESS] 140 / 5000
[PROGRESS] 160 / 5000
[PROGRESS] 180 / 5000
[AUTO-SAVE] 200 data backed up.
[PROGRESS] 200 / 5000
[PROGRESS] 220 / 5000
[PROGRESS] 240 / 5000
[PROGRESS] 260 / 5000
[PROGRESS] 280 / 5000
[AUTO-SAVE] 300 data backed up.
[PROGRESS] 300 / 5000
[PROGRESS] 320 / 5000
[PROGRESS] 340 / 5000
[PROGRESS] 360 / 5000
[PROGRESS] 380 / 5000
[AUTO-SAVE] 400 data backed up.
[PROGRESS] 400 / 5000
[PROGRESS] 420 / 5000
[PROGRESS] 440 / 5000
[PROGRESS] 460 / 5000
[PROGRESS] 480 / 5000
[AUTO-SAVE] 500 data backed up.
[PROGRESS] 500 / 5000
[PROGRESS] 520 / 5000
[PROGRESS] 540 / 5000
[PROGRESS] 560 / 5000
[PROGRESS] 580 / 5000
[AUTO-SAVE] 600 data backed up.
[PROGRESS] 600 / 5000
[PROGRESS] 620 / 5000
[PROGRESS] 640 / 5000
[PROGRESS] 660 / 5000
[PROGRESS] 680 / 5000
[AUTO-SAVE] 700 data backe

okee bulan agustus 

In [ ]:
import requests
import time
import pandas as pd

# --- Konfigurasi ---
BASE = "https://api.twitterapi.io/twitter/tweet/advanced_search"
HEAD = {"X-API-Key": "xxxx"}  # Ganti dengan key aktif
query = "(saham OR investasi OR BEI) lang:in since_time:1754006400 until_time:1756684800 -filter:retweets"
TARGET_JUMLAH = 5000
MAX_RETRIES = 5


def main():
    semua = []
    seen = set()
    cursor = ""
    retry_count = 0
    last_save_count = 0

    session = requests.Session()
    session.headers.update(HEAD)

    print("[INFO] Memulai ekstraksi data IHSG...")

    while len(semua) < TARGET_JUMLAH:
        params = {"query": query, "queryType": "Latest"}
        if cursor:
            params["cursor"] = cursor

        try:
            resp = session.get(BASE, params=params, timeout=15)
            if resp.status_code != 200:
                if resp.status_code == 429:
                    retry_after = resp.headers.get("Retry-After")
                    wait = (
                        int(retry_after)
                        if retry_after and str(retry_after).isdigit()
                        else 15
                    )
                    print(f"[RATE-LIMIT] Tunggu {wait}s...")
                    time.sleep(wait)
                    continue
                retry_count += 1
                print(
                    f"[WARNING] Error {resp.status_code} ({retry_count}/{MAX_RETRIES})"
                )
                time.sleep(10)
                if retry_count >= MAX_RETRIES:
                    print("[ERROR] Max retries. Stop.")
                    break
                continue
            r = resp.json()
            retry_count = 0
        except Exception as e:
            retry_count += 1
            print(f"[ERROR] {e} ({retry_count}/{MAX_RETRIES})")
            time.sleep(5)
            if retry_count >= MAX_RETRIES:
                print("[ERROR] Connection failed. Stop.")
                break
            continue

        tweets = r.get("tweets", [])
        if not tweets:
            print("[INFO] No more tweets.")
            break

        for t in tweets:
            tid = t.get("id")
            if not tid or tid in seen:
                continue
            seen.add(tid)
            user = t.get("author") or t.get("user") or {}

            # ---- EKSTRAKSI KONSISTEN ----
            username = (
                t.get("username") or user.get("userName") or user.get("screen_name")
            )
            name = t.get("name") or user.get("name")
            followers = (
                t.get("followers")
                if t.get("followers") is not None
                else user.get("followers")
            )
            verified = (
                t.get("verified")
                if t.get("verified") is not None
                else user.get("isBlueVerified")
            )

            row = {
                "id": tid,
                "created_at": t.get("createdAt"),
                "username": username,
                "name": name,
                "followers": followers,
                "verified": verified,
                "text": t.get("text"),
                "views": t.get("views"),
                "likes": t.get("likeCount"),
                "replies": t.get("replyCount"),
            }
            semua.append(row)

            if len(semua) >= TARGET_JUMLAH:
                break

        # Auto-save dengan encoding UTF-8-SIG untuk kompatibilitas penuh
        if len(semua) - last_save_count >= 100:
            pd.DataFrame(semua).to_csv(
                "dataset_ihsg_backup kalimat lain agustus .csv",
                index=False,
                encoding="utf-8-sig",
            )
            print(f"[AUTO-SAVE] {len(semua)} data backed up.")
            last_save_count = len(semua)

        print(f"[PROGRESS] {len(semua)} / {TARGET_JUMLAH}")

        if not r.get("has_next_page"):
            break
        cursor = r.get("next_cursor")
        if not cursor:
            break
        time.sleep(2)

    # Finalisasi
    df = pd.DataFrame(semua)
    if not df.empty:
        df = df.drop_duplicates(subset=["id"])
        df.to_csv("dataset_kalimat lain agustus.csv", index=False, encoding="utf-8-sig")
        print(f"\n[SUCCESS] {len(df)} unique tweets saved.")
    else:
        print("\n[WARNING] No data.")


if __name__ == "__main__":
    main()

[INFO] Memulai ekstraksi data IHSG...
[PROGRESS] 20 / 8000
[PROGRESS] 40 / 8000
[PROGRESS] 60 / 8000
[PROGRESS] 80 / 8000
[AUTO-SAVE] 100 data backed up.
[PROGRESS] 100 / 8000
[PROGRESS] 120 / 8000
[PROGRESS] 140 / 8000
[PROGRESS] 160 / 8000
[PROGRESS] 180 / 8000
[AUTO-SAVE] 200 data backed up.
[PROGRESS] 200 / 8000
[PROGRESS] 220 / 8000
[PROGRESS] 240 / 8000
[PROGRESS] 260 / 8000
[PROGRESS] 280 / 8000
[AUTO-SAVE] 300 data backed up.
[PROGRESS] 300 / 8000
[PROGRESS] 320 / 8000
[PROGRESS] 340 / 8000
[PROGRESS] 360 / 8000
[PROGRESS] 380 / 8000
[AUTO-SAVE] 400 data backed up.
[PROGRESS] 400 / 8000
[PROGRESS] 420 / 8000
[PROGRESS] 440 / 8000
[PROGRESS] 460 / 8000
[PROGRESS] 480 / 8000
[AUTO-SAVE] 500 data backed up.
[PROGRESS] 500 / 8000
[PROGRESS] 520 / 8000
[PROGRESS] 540 / 8000
[PROGRESS] 560 / 8000
[PROGRESS] 580 / 8000
[AUTO-SAVE] 600 data backed up.
[PROGRESS] 600 / 8000
[PROGRESS] 620 / 8000
[PROGRESS] 640 / 8000
[PROGRESS] 660 / 8000
[PROGRESS] 680 / 8000
[AUTO-SAVE] 700 data backe

Agustus part 2 hanya 2 konteks saja 

In [ ]:
import requests
import time
import pandas as pd

# --- Konfigurasi ---
BASE = "https://api.twitterapi.io/twitter/tweet/advanced_search"
HEAD = {"X-API-Key": "xxxxx"}  # Ganti dengan key aktif
query = "(saham OR BEI) lang:in since_time:1751328000 until_time:1754006400 -filter:retweets"
TARGET_JUMLAH = 5000
MAX_RETRIES = 5


def main():
    semua = []
    seen = set()
    cursor = ""
    retry_count = 0
    last_save_count = 0

    session = requests.Session()
    session.headers.update(HEAD)

    print("[INFO] Memulai ekstraksi data IHSG...")

    while len(semua) < TARGET_JUMLAH:
        params = {"query": query, "queryType": "Latest"}
        if cursor:
            params["cursor"] = cursor

        try:
            resp = session.get(BASE, params=params, timeout=15)
            if resp.status_code != 200:
                if resp.status_code == 429:
                    retry_after = resp.headers.get("Retry-After")
                    wait = (
                        int(retry_after)
                        if retry_after and str(retry_after).isdigit()
                        else 15
                    )
                    print(f"[RATE-LIMIT] Tunggu {wait}s...")
                    time.sleep(wait)
                    continue
                retry_count += 1
                print(
                    f"[WARNING] Error {resp.status_code} ({retry_count}/{MAX_RETRIES})"
                )
                time.sleep(10)
                if retry_count >= MAX_RETRIES:
                    print("[ERROR] Max retries. Stop.")
                    break
                continue
            r = resp.json()
            retry_count = 0
        except Exception as e:
            retry_count += 1
            print(f"[ERROR] {e} ({retry_count}/{MAX_RETRIES})")
            time.sleep(5)
            if retry_count >= MAX_RETRIES:
                print("[ERROR] Connection failed. Stop.")
                break
            continue

        tweets = r.get("tweets", [])
        if not tweets:
            print("[INFO] No more tweets.")
            break

        for t in tweets:
            tid = t.get("id")
            if not tid or tid in seen:
                continue
            seen.add(tid)
            user = t.get("author") or t.get("user") or {}

            # ---- EKSTRAKSI KONSISTEN ----
            username = (
                t.get("username") or user.get("userName") or user.get("screen_name")
            )
            name = t.get("name") or user.get("name")
            followers = (
                t.get("followers")
                if t.get("followers") is not None
                else user.get("followers")
            )
            verified = (
                t.get("verified")
                if t.get("verified") is not None
                else user.get("isBlueVerified")
            )

            row = {
                "id": tid,
                "created_at": t.get("createdAt"),
                "username": username,
                "name": name,
                "followers": followers,
                "verified": verified,
                "text": t.get("text"),
                "views": t.get("views"),
                "likes": t.get("likeCount"),
                "replies": t.get("replyCount"),
            }
            semua.append(row)

            if len(semua) >= TARGET_JUMLAH:
                break

        # Auto-save dengan encoding UTF-8-SIG untuk kompatibilitas penuh
        if len(semua) - last_save_count >= 100:
            pd.DataFrame(semua).to_csv(
                "dataset_ihsg_backup kalimat lain agustus .csv",
                index=False,
                encoding="utf-8-sig",
            )
            print(f"[AUTO-SAVE] {len(semua)} data backed up.")
            last_save_count = len(semua)

        print(f"[PROGRESS] {len(semua)} / {TARGET_JUMLAH}")

        if not r.get("has_next_page"):
            break
        cursor = r.get("next_cursor")
        if not cursor:
            break
        time.sleep(2)

    # Finalisasi
    df = pd.DataFrame(semua)
    if not df.empty:
        df = df.drop_duplicates(subset=["id"])
        df.to_csv("dataset_kalimat lain agustus.csv", index=False, encoding="utf-8-sig")
        print(f"\n[SUCCESS] {len(df)} unique tweets saved.")
    else:
        print("\n[WARNING] No data.")


if __name__ == "__main__":
    main()

Pada bulan juli

In [ ]:
import requests
import time
import pandas as pd

# --- Konfigurasi ---
BASE = "https://api.twitterapi.io/twitter/tweet/advanced_search"
HEAD = {"X-API-Key": "xxxx"}  # Ganti dengan key aktif
query = "(saham OR BEI) lang:in since_time:1751328000 until_time:1754006400 -filter:retweets"
TARGET_JUMLAH = 5000
MAX_RETRIES = 5


def main():
    semua = []
    seen = set()
    cursor = ""
    retry_count = 0
    last_save_count = 0

    session = requests.Session()
    session.headers.update(HEAD)

    print("[INFO] Memulai ekstraksi data IHSG...")

    while len(semua) < TARGET_JUMLAH:
        params = {"query": query, "queryType": "Latest"}
        if cursor:
            params["cursor"] = cursor

        try:
            resp = session.get(BASE, params=params, timeout=15)
            if resp.status_code != 200:
                if resp.status_code == 429:
                    retry_after = resp.headers.get("Retry-After")
                    wait = (
                        int(retry_after)
                        if retry_after and str(retry_after).isdigit()
                        else 15
                    )
                    print(f"[RATE-LIMIT] Tunggu {wait}s...")
                    time.sleep(wait)
                    continue
                retry_count += 1
                print(
                    f"[WARNING] Error {resp.status_code} ({retry_count}/{MAX_RETRIES})"
                )
                time.sleep(10)
                if retry_count >= MAX_RETRIES:
                    print("[ERROR] Max retries. Stop.")
                    break
                continue
            r = resp.json()
            retry_count = 0
        except Exception as e:
            retry_count += 1
            print(f"[ERROR] {e} ({retry_count}/{MAX_RETRIES})")
            time.sleep(5)
            if retry_count >= MAX_RETRIES:
                print("[ERROR] Connection failed. Stop.")
                break
            continue

        tweets = r.get("tweets", [])
        if not tweets:
            print("[INFO] No more tweets.")
            break

        for t in tweets:
            tid = t.get("id")
            if not tid or tid in seen:
                continue
            seen.add(tid)
            user = t.get("author") or t.get("user") or {}

            # ---- EKSTRAKSI KONSISTEN ----
            username = (
                t.get("username") or user.get("userName") or user.get("screen_name")
            )
            name = t.get("name") or user.get("name")
            followers = (
                t.get("followers")
                if t.get("followers") is not None
                else user.get("followers")
            )
            verified = (
                t.get("verified")
                if t.get("verified") is not None
                else user.get("isBlueVerified")
            )

            row = {
                "id": tid,
                "created_at": t.get("createdAt"),
                "username": username,
                "name": name,
                "followers": followers,
                "verified": verified,
                "text": t.get("text"),
                "views": t.get("views"),
                "likes": t.get("likeCount"),
                "replies": t.get("replyCount"),
            }
            semua.append(row)

            if len(semua) >= TARGET_JUMLAH:
                break

        # Auto-save dengan encoding UTF-8-SIG untuk kompatibilitas penuh
        if len(semua) - last_save_count >= 100:
            pd.DataFrame(semua).to_csv(
                "dataset_ihsg_backup kalimat lain agustus .csv",
                index=False,
                encoding="utf-8-sig",
            )
            print(f"[AUTO-SAVE] {len(semua)} data backed up.")
            last_save_count = len(semua)

        print(f"[PROGRESS] {len(semua)} / {TARGET_JUMLAH}")

        if not r.get("has_next_page"):
            break
        cursor = r.get("next_cursor")
        if not cursor:
            break
        time.sleep(2)

    # Finalisasi
    df = pd.DataFrame(semua)
    if not df.empty:
        df = df.drop_duplicates(subset=["id"])
        df.to_csv("dataset_kalimat lain agustus.csv", index=False, encoding="utf-8-sig")
        print(f"\n[SUCCESS] {len(df)} unique tweets saved.")
    else:
        print("\n[WARNING] No data.")


if __name__ == "__main__":
    main()

[INFO] Memulai ekstraksi data IHSG...
[PROGRESS] 20 / 5000
[PROGRESS] 40 / 5000
[PROGRESS] 60 / 5000
[PROGRESS] 80 / 5000
[AUTO-SAVE] 100 data backed up.
[PROGRESS] 100 / 5000
[PROGRESS] 120 / 5000
[PROGRESS] 140 / 5000
[PROGRESS] 160 / 5000
[PROGRESS] 180 / 5000
[AUTO-SAVE] 200 data backed up.
[PROGRESS] 200 / 5000
[PROGRESS] 220 / 5000
[PROGRESS] 240 / 5000
[PROGRESS] 260 / 5000
[PROGRESS] 280 / 5000
[AUTO-SAVE] 300 data backed up.
[PROGRESS] 300 / 5000
[PROGRESS] 320 / 5000
[PROGRESS] 340 / 5000
[PROGRESS] 360 / 5000
[PROGRESS] 380 / 5000
[AUTO-SAVE] 400 data backed up.
[PROGRESS] 400 / 5000
[PROGRESS] 420 / 5000
[PROGRESS] 440 / 5000
[PROGRESS] 460 / 5000
[PROGRESS] 480 / 5000
[AUTO-SAVE] 500 data backed up.
[PROGRESS] 500 / 5000
[PROGRESS] 520 / 5000
[PROGRESS] 540 / 5000
[PROGRESS] 560 / 5000
[PROGRESS] 580 / 5000
[AUTO-SAVE] 600 data backed up.
[PROGRESS] 600 / 5000
[PROGRESS] 620 / 5000
[PROGRESS] 640 / 5000
[PROGRESS] 660 / 5000
[PROGRESS] 680 / 5000
[AUTO-SAVE] 700 data backe

Bulan Juni 

In [ ]:
import requests
import time
import pandas as pd

# --- Konfigurasi ---
BASE = "https://api.twitterapi.io/twitter/tweet/advanced_search"
HEAD = {"X-API-Key": "xxxx"}  # Ganti dengan key aktif
query = "(saham OR BEI) lang:in since_time:1748736000 until_time:1751328000 -filter:retweets"
TARGET_JUMLAH = 200
MAX_RETRIES = 5


def main():
    semua = []
    seen = set()
    cursor = ""
    retry_count = 0
    last_save_count = 0

    session = requests.Session()
    session.headers.update(HEAD)

    print("[INFO] Memulai ekstraksi data IHSG...")

    while len(semua) < TARGET_JUMLAH:
        params = {"query": query, "queryType": "Latest"}
        if cursor:
            params["cursor"] = cursor

        try:
            resp = session.get(BASE, params=params, timeout=15)
            if resp.status_code != 200:
                if resp.status_code == 429:
                    retry_after = resp.headers.get("Retry-After")
                    wait = (
                        int(retry_after)
                        if retry_after and str(retry_after).isdigit()
                        else 15
                    )
                    print(f"[RATE-LIMIT] Tunggu {wait}s...")
                    time.sleep(wait)
                    continue
                retry_count += 1
                print(
                    f"[WARNING] Error {resp.status_code} ({retry_count}/{MAX_RETRIES})"
                )
                time.sleep(10)
                if retry_count >= MAX_RETRIES:
                    print("[ERROR] Max retries. Stop.")
                    break
                continue
            r = resp.json()
            retry_count = 0
        except Exception as e:
            retry_count += 1
            print(f"[ERROR] {e} ({retry_count}/{MAX_RETRIES})")
            time.sleep(5)
            if retry_count >= MAX_RETRIES:
                print("[ERROR] Connection failed. Stop.")
                break
            continue

        tweets = r.get("tweets", [])
        if not tweets:
            print("[INFO] No more tweets.")
            break

        for t in tweets:
            tid = t.get("id")
            if not tid or tid in seen:
                continue
            seen.add(tid)
            user = t.get("author") or t.get("user") or {}

            # ---- EKSTRAKSI KONSISTEN ----
            username = (
                t.get("username") or user.get("userName") or user.get("screen_name")
            )
            name = t.get("name") or user.get("name")
            followers = (
                t.get("followers")
                if t.get("followers") is not None
                else user.get("followers")
            )
            verified = (
                t.get("verified")
                if t.get("verified") is not None
                else user.get("isBlueVerified")
            )

            row = {
                "id": tid,
                "created_at": t.get("createdAt"),
                "username": username,
                "name": name,
                "followers": followers,
                "verified": verified,
                "text": t.get("text"),
                "views": t.get("views"),
                "likes": t.get("likeCount"),
                "replies": t.get("replyCount"),
            }
            semua.append(row)

            if len(semua) >= TARGET_JUMLAH:
                break

        # Auto-save dengan encoding UTF-8-SIG untuk kompatibilitas penuh
        if len(semua) - last_save_count >= 100:
            pd.DataFrame(semua).to_csv(
                "dataset_ihsg_backup kalimat lain agustus .csv",
                index=False,
                encoding="utf-8-sig",
            )
            print(f"[AUTO-SAVE] {len(semua)} data backed up.")
            last_save_count = len(semua)

        print(f"[PROGRESS] {len(semua)} / {TARGET_JUMLAH}")

        if not r.get("has_next_page"):
            break
        cursor = r.get("next_cursor")
        if not cursor:
            break
        time.sleep(2)

    # Finalisasi
    df = pd.DataFrame(semua)
    if not df.empty:
        df = df.drop_duplicates(subset=["id"])
        df.to_csv("dataset_kalimat lain agustus.csv", index=False, encoding="utf-8-sig")
        print(f"\n[SUCCESS] {len(df)} unique tweets saved.")
    else:
        print("\n[WARNING] No data.")


if __name__ == "__main__":
    main()

[INFO] Memulai ekstraksi data IHSG...
[PROGRESS] 20 / 5000
[PROGRESS] 40 / 5000
[PROGRESS] 60 / 5000
[PROGRESS] 80 / 5000
[AUTO-SAVE] 100 data backed up.
[PROGRESS] 100 / 5000
[PROGRESS] 120 / 5000
[PROGRESS] 140 / 5000
[PROGRESS] 160 / 5000
[PROGRESS] 180 / 5000
[AUTO-SAVE] 200 data backed up.
[PROGRESS] 200 / 5000
[PROGRESS] 220 / 5000
[PROGRESS] 240 / 5000
[PROGRESS] 260 / 5000
[PROGRESS] 280 / 5000
[AUTO-SAVE] 300 data backed up.
[PROGRESS] 300 / 5000
[PROGRESS] 320 / 5000
[PROGRESS] 340 / 5000
[PROGRESS] 360 / 5000
[PROGRESS] 380 / 5000
[AUTO-SAVE] 400 data backed up.
[PROGRESS] 400 / 5000
[PROGRESS] 420 / 5000
[PROGRESS] 440 / 5000
[PROGRESS] 460 / 5000
[PROGRESS] 480 / 5000
[AUTO-SAVE] 500 data backed up.
[PROGRESS] 500 / 5000
[PROGRESS] 520 / 5000
[PROGRESS] 539 / 5000
[PROGRESS] 559 / 5000
[PROGRESS] 579 / 5000
[PROGRESS] 599 / 5000
[AUTO-SAVE] 619 data backed up.
[PROGRESS] 619 / 5000
[PROGRESS] 639 / 5000
[PROGRESS] 659 / 5000
[PROGRESS] 679 / 5000
[PROGRESS] 699 / 5000
[AUT

banyak yang gak jelas isinya, coba liat lagi 
apa fokus di ihsg saja 


In [ ]:
import requests
import time
import pandas as pd

# --- Konfigurasi ---
BASE = "https://api.twitterapi.io/twitter/tweet/advanced_search"
HEAD = {"X-API-Key": "xxxx"}  # Ganti dengan key aktif
query = "(unjuk rasa ihsg) lang:in since_time:1754006400 until_time:1756684800 -filter:retweets"
TARGET_JUMLAH = 1000
MAX_RETRIES = 5


def main():
    semua = []
    seen = set()
    cursor = ""
    retry_count = 0
    last_save_count = 0

    session = requests.Session()
    session.headers.update(HEAD)

    print("[INFO] Memulai ekstraksi data IHSG...")

    while len(semua) < TARGET_JUMLAH:
        params = {"query": query, "queryType": "Latest"}
        if cursor:
            params["cursor"] = cursor

        try:
            resp = session.get(BASE, params=params, timeout=15)
            if resp.status_code != 200:
                if resp.status_code == 429:
                    retry_after = resp.headers.get("Retry-After")
                    wait = (
                        int(retry_after)
                        if retry_after and str(retry_after).isdigit()
                        else 15
                    )
                    print(f"[RATE-LIMIT] Tunggu {wait}s...")
                    time.sleep(wait)
                    continue
                retry_count += 1
                print(
                    f"[WARNING] Error {resp.status_code} ({retry_count}/{MAX_RETRIES})"
                )
                time.sleep(10)
                if retry_count >= MAX_RETRIES:
                    print("[ERROR] Max retries. Stop.")
                    break
                continue
            r = resp.json()
            retry_count = 0
        except Exception as e:
            retry_count += 1
            print(f"[ERROR] {e} ({retry_count}/{MAX_RETRIES})")
            time.sleep(5)
            if retry_count >= MAX_RETRIES:
                print("[ERROR] Connection failed. Stop.")
                break
            continue

        tweets = r.get("tweets", [])
        if not tweets:
            print("[INFO] No more tweets.")
            break

        for t in tweets:
            tid = t.get("id")
            if not tid or tid in seen:
                continue
            seen.add(tid)
            user = t.get("author") or t.get("user") or {}

            # ---- EKSTRAKSI KONSISTEN ----
            username = (
                t.get("username") or user.get("userName") or user.get("screen_name")
            )
            name = t.get("name") or user.get("name")
            followers = (
                t.get("followers")
                if t.get("followers") is not None
                else user.get("followers")
            )
            verified = (
                t.get("verified")
                if t.get("verified") is not None
                else user.get("isBlueVerified")
            )

            row = {
                "id": tid,
                "created_at": t.get("createdAt"),
                "username": username,
                "name": name,
                "followers": followers,
                "verified": verified,
                "text": t.get("text"),
                "views": t.get("views"),
                "likes": t.get("likeCount"),
                "replies": t.get("replyCount"),
            }
            semua.append(row)

            if len(semua) >= TARGET_JUMLAH:
                break

        # Auto-save dengan encoding UTF-8-SIG untuk kompatibilitas penuh
        if len(semua) - last_save_count >= 100:
            pd.DataFrame(semua).to_csv(
                "dataset_ihsg_backup demo ihsg.csv", index=False, encoding="utf-8-sig"
            )
            print(f"[AUTO-SAVE] {len(semua)} data backed up.")
            last_save_count = len(semua)

        print(f"[PROGRESS] {len(semua)} / {TARGET_JUMLAH}")

        if not r.get("has_next_page"):
            break
        cursor = r.get("next_cursor")
        if not cursor:
            break
        time.sleep(2)

    # Finalisasi
    df = pd.DataFrame(semua)
    if not df.empty:
        df = df.drop_duplicates(subset=["id"])
        df.to_csv("dataset_kalimat demo ihsg.csv", index=False, encoding="utf-8-sig")
        print(f"\n[SUCCESS] {len(df)} unique tweets saved.")
    else:
        print("\n[WARNING] No data.")


if __name__ == "__main__":
    main()

[INFO] Memulai ekstraksi data IHSG...
[PROGRESS] 20 / 2000
[PROGRESS] 40 / 2000
[PROGRESS] 49 / 2000
[INFO] No more tweets.

[SUCCESS] 49 unique tweets saved.
